In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.colorbar as colorbar

from typing import Tuple, Literal
from anndata import AnnData
from itertools import product
from tqdm.auto import tqdm

from scipy.stats import pearsonr
import textwrap

#### dataset publication: https://www.nature.com/articles/s41467-019-14118-w

In [ ]:
for mod in [np, pd, sc, gp]:
    print(mod, mod.__version__)

In [ ]:
def get_genesets(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype_signature: str,
    direction: str,
    threshold_pval: float,
    threshold_lfc: float,
    min_size: int = None,
    max_size: int = None,
) -> Tuple[dict[str, list[str]], pd.DataFrame]:

    # construct signature gene set
    select = ((df_hcd_all.adj_p_value <= threshold_pval) & (df_hcd_all.celltype == celltype_signature))
    if direction == "upregulated":
        select = select & (df_hcd_all.log_fc >= threshold_lfc)
    elif direction == "downregulated":
        select = select & (df_hcd_all.log_fc <= threshold_lfc)
    elif direction == "both":
        select = select & (df_hcd_all.log_fc.abs() >= threshold_lfc)         
    
    df_hcd_ct = df_hcd_all.loc[select]
    gene_set_dict = {}
    gene_set_df = pd.DataFrame()
    for cytokine_i, cytokine in enumerate(df_hcd_ct.cytokine.unique()):
        gene_set = df_hcd_ct.loc[df_hcd_ct.cytokine == cytokine].gene.values
        gene_set_shared = np.intersect1d(gene_set, adata.var_names)
        gene_set_df.loc[cytokine_i, "cytokine"] = cytokine
        gene_set_df.loc[cytokine_i, "num_genes_signature"] = len(gene_set)
        gene_set_df.loc[cytokine_i, "num_shared_genes_signature"] = len(gene_set_shared)
        gene_set_df.loc[cytokine_i, "frac_shared_genes_signature"] = len(gene_set_shared) / len(gene_set)

        if (len(gene_set_shared) < min_size) or (len(gene_set_shared) > max_size): 
            continue
        
        gene_set_dict[cytokine] = gene_set_shared

    return gene_set_dict, gene_set_df


def compute_s2n(adata: AnnData, contrast_column: str, condition_1: str, condition_2: str) -> (pd.DataFrame, pd.DataFrame):
    """
    Compute the signal-to-noise ratio (S2N) for each gene between two conditions in an AnnData object.

    Parameters:
    - adata: AnnData object with gene expression data.
    - contrast_column: Key in `adata.obs` indicating the condition labels (e.g. "disease_state").
    - condition_1: Name of the first condition (e.g., "flare").
    - condition_2: Name of the second condition (e.g., "healthy").
    
    Returns:
    - s2n_scores: pandas Series of S2N values indexed by gene names.
    """
    # Select cells for each condition
    group1 = adata[adata.obs[contrast_column] == condition_1]
    group2 = adata[adata.obs[contrast_column] == condition_2]

    # number of cells per condition
    num_cells_1 = group1.shape[0]
    num_cells_2 = group2.shape[0]
    
    # Get expression matrices
    X1 = group1.X.toarray() if hasattr(group1.X, "toarray") else group1.X
    X2 = group2.X.toarray() if hasattr(group2.X, "toarray") else group2.X

    # Compute mean and std per gene
    mu1 = np.mean(X1, axis=0)
    mu2 = np.mean(X2, axis=0)
    sigma1 = np.std(X1, axis=0, ddof=1)
    sigma2 = np.std(X2, axis=0, ddof=1)

    # Compute S2N
    s2n = (mu1 - mu2) / (sigma1 + sigma2 + 1e-8)  # epsilon to avoid division by zero

    num_cells = pd.DataFrame(index=[f"{condition_1}_vs_{condition_2}"], columns=["num_cells_1", "num_cells_2"], data=[[num_cells_1, num_cells_2]])
    stats = pd.DataFrame(s2n, index=adata.var_names, columns=[f"{condition_1}_vs_{condition_2}"])
    
    return stats, num_cells

  
def compute_ranking_statistic(adata: AnnData, contrast_column: str, contrasts: list[Tuple[str, str]]) -> (pd.DataFrame, pd.DataFrame):
    rnk_stats, num_cells = [], []
    for condition in contrasts:
        _rnk_stats, _num_cells = compute_s2n(
            adata,
            contrast_column=contrast_column,
            condition_1=condition[0],
            condition_2=condition[1]
        )
        rnk_stats.append(_rnk_stats)
        num_cells.append(_num_cells)
    return pd.concat(rnk_stats, axis=1), pd.concat(num_cells, axis=0)

    
def run_enrichment_test(
    adata: AnnData,
    df_hcd_all: pd.DataFrame,
    celltype: Tuple[str, str] = ("B cell", "B"),
    direction: str = "upregulated",
    threshold_pval: float = 0.01,
    threshold_lfc: float = 1.,
    threshold_expression: float = 0,
    contrast_column: str = "disease_state",
    celltype_column: str = "disease_state",
    contrasts: Tuple[str, str] | list[Tuple[str, str]] = None,
    min_size: int = 10,
    max_size: int = 1000,
    permutation_num: int = 1000,
    weight: float = 1.,
    max_obs_per_condition: int | None = None, # None = all
    seed: int = 2025,
    verbose: bool = True,
    threads: int = 6,
) -> pd.DataFrame:

    if not isinstance(contrasts, list):
        assert isinstance(contrasts, Tuple)
        contrasts = [contrasts]
        
    celltype_adata = celltype[0]
    celltype_signature = celltype[1]

    # filter for cell type
    adata = adata[adata.obs[celltype_column] == celltype_adata]
    
    # filter based on gene expression
    adata = adata[:, adata.X.mean(axis=0) >= threshold_expression]

    # get genesets
    gene_set_dict, gene_set_df = get_genesets(
        adata=adata,
        df_hcd_all=df_hcd_all,
        celltype_signature=celltype_signature,
        direction=direction,
        threshold_pval=threshold_pval,
        threshold_lfc=threshold_lfc,
        min_size=min_size,
        max_size=max_size,
    )

    # return adata, gene_set_dict

    if len(gene_set_dict) == 0:
        print("No gene sets passed filtering criteria.")
        return None
    
    # compute ranking stat
    rnk_stats, num_cells_per_condition = compute_ranking_statistic(adata, contrast_column=contrast_column, contrasts=contrasts)
    results = []
    for contrast_name in rnk_stats.columns:
        
        # format stat so that it can be processed be gseapy
        
        rnk = rnk_stats.loc[:, contrast_name]\
            .replace([np.inf, -np.inf], np.nan)\
            .dropna()\
            .sort_values(ascending=False)\
            .to_frame()\
            .rename({contrast_name: 1}, axis=1)\
            .rename_axis("0")
        
        # run enrichment
        gp_res = gp.prerank(
            rnk=rnk,
            gene_sets=gene_set_dict,
            min_size=min_size,
            max_size=max_size,
            permutation_num=permutation_num,
            weight=weight,
            outdir=None,
            seed=seed,
            verbose=verbose,
            threads=threads,
        )
        _res = gp_res.res2d
        _res.loc[:, "contrast"] = contrast_name
        _res.loc[:, "num_cells_1"] = num_cells_per_condition.loc[contrast_name, "num_cells_1"]
        _res.loc[:, "num_cells_2"] = num_cells_per_condition.loc[contrast_name, "num_cells_2"]
        _res.loc[:, "percent_duplicate_ranking_stats"] = (rnk.duplicated(keep="first").sum() / rnk.shape[0]) * 100
        
        results.append(_res)

    # combine results and save hyperparams
    results = pd.concat(results, axis=0, ignore_index=True)
    results.loc[:, "celltype_adata"] = celltype_adata
    results.loc[:, "celltype_signature"] = celltype_signature
    results.loc[:, "celltype_combo"] = f"{celltype_adata} ({celltype_signature})"
    results.loc[:, "direction"] = direction
    results.loc[:, "threshold_pval"] = threshold_pval
    results.loc[:, "threshold_lfc"] = threshold_lfc
    results.loc[:, "threshold_expression"] = threshold_expression
    results.loc[:, "min_size"] = min_size
    results.loc[:, "max_size"] = max_size
    results.loc[:, "permutation_num"] = permutation_num
    results.loc[:, "weight"] = weight
    results.loc[:, "seed"] = seed
    results.loc[:, "threads"] = threads

    results.rename({"Term": "cytokine"}, inplace=True, axis=1)
    results = pd.merge(results, gene_set_df, on="cytokine")
    
    return results

In [ ]:
fig_path = "/Users/soeren.becker/repos/HumanCytokineDict/notebooks/08_usecase_MS/figures_manuscript_programs"
os.makedirs(fig_path, exist_ok=True)

In [ ]:
# pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/notebooks/larsen/enrichment_MS/polarization_of_programs_interpretability_new.csv", sep=';', decimal=',')#, index_col=0)

### Load and prepare program signatures

In [ ]:
df_drvi = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/notebooks/04_gene_programs/Amir/significant_effects_df.csv", index_col=0)
interpretability_scores = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/drvi/interpretability_scores/combined_score_traverse_effect_pos.csv", index_col=0)
de_anndata = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/notebooks/04_gene_programs/Amir/de_anndata_combined_v1.h5ad")
all_effects_df = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/notebooks/04_gene_programs/Amir/all_effects.csv", index_col=0)
significant_effects_df = all_effects_df[all_effects_df['is_significant'] == 1]
significant_effects = significant_effects_df.title.unique()
sig_anndata = de_anndata[:, de_anndata.var.title.isin(significant_effects)]
sig_scores = interpretability_scores.loc[:, sig_anndata.var.original_dim_id.values.astype(str)]

dim2title = dict(de_anndata.var.loc[:, ["original_dim_id", "title"]].astype(str).values)
title2process = dict(all_effects_df.loc[:, ["title", "process_id"]].values)

_df = sig_scores.rename(dim2title, axis=1).rename(title2process, axis=1)

df_prepared = pd.melt(
    _df.reset_index(), 
    id_vars='index', 
    var_name='Category', 
    value_name='Value',
).rename({"index": "gene", "Category": "process_id", "Value": "log_fc"}, axis=1)

df_prepared = df_prepared.loc[df_prepared.log_fc>0]
df_prepared2 = df_drvi.loc[:, ["process_id", "cell_type"]].drop_duplicates()
df_drvi_prepared = df_prepared.merge(df_prepared2, on='process_id', how='left')
df_drvi_prepared.loc[:, "adj_p_value"] = 1e-20 # arbitrary, just need a column with this name
df_drvi_prepared = df_drvi_prepared.rename({"process_id": "cytokine", "cell_type": "celltype"}, axis=1)
df_drvi_prepared

### Load dataset

In [ ]:
adata = sc.read_h5ad("/Users/soeren.becker/repos/HumanCytokineDict/data/use_case_datasets/08_Multiple_Sclerose/Schafflick20_MS_CSF.h5ad")
invalid_labels = [
    "Mono Doublet",
    "contamination1",
    "B cell doublets",
    "doublet",
]
adata = adata[~adata.obs.labels.isin(invalid_labels)]

In [ ]:
hcd_var_names = df_drvi_prepared.gene.unique()
print(len(hcd_var_names))

usecase_var_names = adata.var_names.values
print(len(usecase_var_names))

print(len(np.intersect1d(usecase_var_names, hcd_var_names)))

In [ ]:
sorted(np.array(list(adata.obs.labels.unique())))

In [ ]:
sorted(df_drvi_prepared.celltype.unique())

In [ ]:
celltype_combos = ( # cell type in RA dataset, celltype of gene set signature
    ### T cells ###
    ('CD4', "CD4 T cell"),
    ('CD8a', "CD8 T cell"),
    ('CD8n', "CD8 T cell"),
    ('Tregs', "Treg"),
    ### Monocytes ###
    ('Mono', "CD14 Mono"),
    ('ncMono', "CD16 Mono"),
    ### B cells ###
    ('B1', "B cell"), 
    ('B2', "B cell"),
    ### Natural killer cells ###
    ('NK1', 'NK CD56hi'),
    ('NK2', 'NK CD56hi'),
    ('NK1', 'NK CD56low'),
    ('NK2', 'NK CD56low'),
    ### dendritic cells ###
    ('mDC1', "cDC"),
    ('mDC2', "cDC"),
)

In [ ]:
# fontsize
fs = 6

### Run enrichment analysis on CSF

In [ ]:
all_results_csf = []

for threshold_expression in [0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:

    shared_kwargs_csf= {
        "threshold_pval": 0.05, # arbitrary, all programs will pass
        "threshold_lfc": 0.5,
        "threshold_expression": threshold_expression,
        "direction": "upregulated",
        "min_size": 8,
        "max_size": 1_000,
        "permutation_num": 20_000,
        "verbose": True,
        "adata": adata[adata.obs.CSF == "True"],
        "df_hcd_all": df_drvi_prepared,
    }
    
    for celltype_combo_k, celltype_combo in enumerate(celltype_combos):
    
        results_csf = run_enrichment_test(
            celltype=celltype_combo,
            contrast_column="MS",
            celltype_column="labels",
            contrasts=[("True", "False")],
            **shared_kwargs_csf,
        )
        if results_csf is not None:
            all_results_csf.append(results_csf)

all_results_csf = pd.concat(all_results_csf, axis=0)
all_results_csf.loc[:, "CSF"] = True
all_results_csf.to_csv("./results/all_results_enrichment_programs_csf.csv")

### Run enrichment analysis on blood

In [ ]:
all_results_blood = []

for threshold_expression in [0.05, 0.06, 0.07, 0.08, 0.09, 0.1]:

    shared_kwargs_blood= {
        "threshold_pval": 0.05, # arbitrary, all programs will pass,
        "threshold_lfc": 0.5,
        "threshold_expression": threshold_expression,
        "direction": "upregulated",
        "min_size": 8,
        "max_size": 1_000,
        "permutation_num": 20_000,
        "verbose": True,
        "adata": adata[adata.obs.CSF == "False"],
        "df_hcd_all": df_drvi_prepared,
    }
    
    for celltype_combo_k, celltype_combo in enumerate(celltype_combos):
    
        results_blood = run_enrichment_test(
            celltype=celltype_combo,
            contrast_column="MS",
            celltype_column="labels",
            contrasts=[("True", "False")],
            **shared_kwargs_blood,
        )
        
        all_results_blood.append(results_blood)

all_results_blood = pd.concat(all_results_blood, axis=0)
all_results_blood.loc[:, "CSF"] = False
all_results_blood.to_csv("./results/all_results_enrichment_programs_blood.csv")

In [ ]:
program_colors = {
    'CE': '#8dd3c7',
    'CS': '#ffffb3',
    'ES': '#bebada',
    'FC': '#fb8072',
    'GR': '#80b1d3',
    'II': '#fdb462',
    'IT': '#b3de69',
    'MD': '#fccde5',
    'PD': '#d9d9d9',
    'SN': '#bc80bd',
    'TR': '#ccebc5'
}

def format_celltype_combo_labels(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [format_celltype_combo_labels(xx) for xx in x]
    return x.split(" (")[0]

def format_program_labels(x):
    if isinstance(x, (list, np.ndarray, pd.Index)):
        return [format_program_labels(xx) for xx in x]
    return x.split(" ")[1]

def find_switch_indices(lst):
    """Return indices where the value changes from the previous one."""
    return [0] + [i for i in range(1, len(lst)) if lst[i] != lst[i - 1]]


def _check_robustness_fractions(
    df_pivot, 
    threshold_qval=0.1, # adjusted p value
    threshold_valid=0.5, # fraction of results required to even consider this condition. I.e. if the test only ran for one set of thresholds, then it is not very robust.
    threshold_below_alpha=0.75, # fraction of results that need to be significant
):
    n_total = np.prod(df_pivot.shape)
    n_valid = n_total - df_pivot.isna().sum().sum()
    n_below_alpha = (df_pivot < threshold_qval).sum().sum() # number of results below pval threshold, i.e., number of significant results
    frac_valid_results = (n_valid / n_total)
    frac_pval_below_alpha = (n_below_alpha / n_valid) # fraction of significant results relative to valid results
    is_robust = (frac_pval_below_alpha > threshold_below_alpha) & (frac_valid_results > threshold_valid)
    return frac_valid_results, frac_pval_below_alpha, is_robust


def check_robustness(
    all_results,
    THRESHOLD_QVAL = 0.1,
    THRESHOLD_VALID = 0.5,
    THRESHOLD_BELOW_ALPHA = 0.9,
):
    
    all_thresholds_expression = all_results.threshold_expression.sort_values(ascending = False).unique()
    all_thresholds_lfc = sorted(all_results.threshold_lfc.unique())
    
    df = pd.DataFrame(index=all_thresholds_expression, columns=all_thresholds_lfc)
    df.index.rename("threshold_expression", inplace=True)
    df.columns.rename("threshold_lfc", inplace=True)

    df2 = df.copy()
    
    robust_results = []
    
    for contrast in tqdm(all_results.contrast.unique()):
        for celltype_combo in all_results.celltype_combo.unique():
            results_ct = all_results.loc[(all_results.celltype_combo == celltype_combo) & (all_results.contrast == contrast)]
            for cytokine in results_ct.cytokine.unique():
                results_ct_cy = results_ct.loc[results_ct.cytokine == cytokine]
                df_pivot = results_ct_cy.pivot(index="threshold_expression", columns="threshold_lfc", values="FDR q-val")
                df_pivot_nes = results_ct_cy.pivot(index="threshold_expression", columns="threshold_lfc", values="NES")
                row_order = df_pivot.index.sort_values(ascending=False)
                with warnings.catch_warnings():
                    warnings.simplefilter(action='ignore', category=FutureWarning)
                    df_combined = pd.concat([df, df_pivot])
                    df_combined2 = pd.concat([df2, df_pivot_nes])
                df_merged = df_combined.combine_first(df_pivot)
                df_merged2 = df_combined2.combine_first(df_pivot_nes)
                df_merged = df_merged.loc[~df_merged.index.duplicated()]
                df_merged2 = df_merged2.loc[~df_merged2.index.duplicated()]
                df_pivot = df_merged.loc[all_thresholds_expression, all_thresholds_lfc].astype(float)
                df_pivot_nes = df_merged2.loc[all_thresholds_expression, all_thresholds_lfc].astype(float)
                frac_valid_results, frac_pval_below_alpha, is_robust = _check_robustness_fractions(
                    df_pivot,
                    threshold_qval=THRESHOLD_QVAL,
                    threshold_valid=THRESHOLD_VALID,
                    threshold_below_alpha=THRESHOLD_BELOW_ALPHA
                )
                if True:
                    if is_robust:
                        # print(df_pivot_nes.max().max(), df_pivot_nes.min().min())
                        if (df_pivot_nes.max().max() > 0) and (df_pivot_nes.min().min() < 0):
                            sns.heatmap(df_pivot, cmap="Reds_r", center=0, vmin=0, vmax=0.1, square=True)
                            plt.show()
                            sns.heatmap(df_pivot_nes, cmap="RdBu_r", center=0, square=True)
                            plt.show()
                        robust_results.append(
                            (
                                celltype_combo, contrast, cytokine,
                                frac_valid_results, frac_pval_below_alpha, is_robust,
                                results_ct_cy.NES.min(), results_ct_cy.NES.max(),
                                THRESHOLD_QVAL,
                                THRESHOLD_BELOW_ALPHA,
                            )
                        )
                else:
                    robust_results.append(
                        (
                            celltype_combo, contrast, cytokine,
                            frac_valid_results, frac_pval_below_alpha, is_robust,
                            results_ct_cy.NES.min(), results_ct_cy.NES.max(),
                            THRESHOLD_QVAL,
                            THRESHOLD_BELOW_ALPHA,
                        )
                    )
        
    robust_results = pd.DataFrame(robust_results).rename(
        {0: "celltype_combo", 1: "contrast", 2: "cytokine", 3: "frac_valid", 4: "frac_significant", 5:"is_robust", 6:"NES_min", 7:"NES_max", 8:"qval_threshold", 9: "threshold_frac_below_alpha"},
        axis=1
    )
    return robust_results


def get_robust_significant_results(
    results,
    alphas,
    threshold_valid,
    threshold_below_alpha,
):

    results_robust = []
    for alpha in alphas:
        results_robust.append(check_robustness(results, THRESHOLD_QVAL=alpha, THRESHOLD_VALID=threshold_valid, THRESHOLD_BELOW_ALPHA=threshold_below_alpha))
    results_robust_all = pd.concat(results_robust)
    results_robust = results_robust_all.loc[results_robust_all.is_robust == True]
    results_robust = results_robust_all.groupby(["contrast", "celltype_combo", "cytokine"])["qval_threshold"].min().to_frame().reset_index()
    
    # results_mean = results.fillna(0).groupby(["contrast", "celltype_combo", "cytokine"])["NES"].mean().to_frame().reset_index()
    results_mean = results.groupby(["contrast", "celltype_combo", "cytokine"])["NES"].mean().to_frame().reset_index()
    results_pivot = results_mean.pivot(
        index="cytokine",
        columns="celltype_combo",
        values="NES",
    )
    df_annot = results_pivot.copy()
    with warnings.catch_warnings():
        warnings.simplefilter(action='ignore', category=FutureWarning)
        df_annot[:] = ""
    for cytokine in df_annot.index:
        for celltype in df_annot.columns:
            qval = results_robust.loc[(results_robust.cytokine == cytokine) & (results_robust.celltype_combo == celltype)].qval_threshold
            if len(qval) != 0:
                qval = qval.values[0]
                if qval == 0.1:
                    df_annot.loc[cytokine, celltype] = "*"
                if qval == 0.05:
                    df_annot.loc[cytokine, celltype] = "**"
                if qval == 0.01:
                    df_annot.loc[cytokine, celltype] = "***"

    return results_pivot, df_annot, results_robust, results_robust_all


def create_heatmap(
    results,
    robust_results,
    celltypes,
    cytokines,
    path = None,
    figsize = None,
):
    df_nes = results.loc[(results.cytokine.isin(cytokines) & results.celltype_combo.isin(celltypes))]\
        .groupby(["celltype_combo", "cytokine"])["NES"].mean()\
        .to_frame()\
        .reset_index()\
        .pivot(index="cytokine", columns="celltype_combo", values="NES")
    
    df_nes = df_nes.loc[:, celltypes[np.isin(celltypes, df_nes.columns)]]
    with warnings.catch_warnings():
        warnings.simplefilter(action='ignore', category=FutureWarning)
        df_annot = df_nes.copy()
        df_annot.loc[:, :] = ""
        
    for program in df_annot.index:
        for celltype in df_annot.columns:
            select = ((robust_results.cytokine == program) & (robust_results.celltype_combo == celltype))
            alpha = robust_results.loc[select, "qval_threshold"]
            if len(alpha) == 0:
                df_annot.loc[program, celltype] = ""
            else:
                assert len(alpha) == 1
                alpha = alpha.values[0]
                if alpha == 0.1:
                    df_annot.loc[program, celltype] = "*"
                elif alpha == 0.05:
                    df_annot.loc[program, celltype] = "**"
                elif alpha == 0.01:
                    df_annot.loc[program, celltype] = "***"
    
    if figsize is None:
        fig, ax = plt.subplots(1, 1, figsize=(0.75, 0.75))
    else:
        fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.set_facecolor("lightgray")
    heatmap_out = sns.heatmap(
        df_nes,
        center=0,
        #vmin=-df_blood.NES.max(),
        #vmax=df_blood.NES.max(),
        cmap="RdBu_r",
        square=True,
        annot=df_annot,
        annot_kws={"fontsize": 6, "family": "Arial"},
        fmt="",
        ax=ax,
        cbar=False,
        cbar_kws={"shrink": 0.5},
        lw=0.5,
        linecolor="white",
    )
    plt.xlabel("")
    plt.ylabel("")
    ax.tick_params(axis='both', labelright=False, labelleft=True, length=0)
    ax.set_xticks(0.5+np.arange(df_nes.shape[1]), format_celltype_combo_labels(df_nes.columns), family="Arial", fontsize=6, rotation=90)
    ax.set_yticks(0.5+np.arange(df_nes.shape[0]), format_program_labels(df_nes.index), family="Arial", fontsize=6)
    if path:
        plt.savefig(path, dpi=400, bbox_inches="tight", pad_inches=0)
    plt.show()
    return df_nes, df_annot

In [ ]:
THRESHOLD_VALID = 1/3
THRESHOLD_BELOW_ALPHA = 2/3
THRESHOLD_EXPRESSION = 0.05
THRESHOLD_DUPLICATES = 0.05

In [ ]:
all_results_blood = pd.read_csv("./results/all_results_enrichment_programs_blood.csv")
all_results_blood = all_results_blood.loc[all_results_blood.threshold_expression >= THRESHOLD_EXPRESSION]

all_results_blood = all_results_blood.loc[all_results_blood.celltype_combo != "NK1 (NK CD56hi)"]
all_results_blood = all_results_blood.loc[all_results_blood.celltype_combo != "NK2 (NK CD56low)"]
print(all_results_blood.loc[all_results_blood.percent_duplicate_ranking_stats < THRESHOLD_DUPLICATES].shape[0] / all_results_blood.shape[0])
all_results_blood = all_results_blood.loc[all_results_blood.percent_duplicate_ranking_stats < THRESHOLD_DUPLICATES]
results_pivot_blood, df_annot_blood, results_robust_blood, results_robust_blood_all = get_robust_significant_results(
    all_results_blood,
    alphas=[0.1, 0.05, 0.01],
    threshold_valid=THRESHOLD_VALID,
    threshold_below_alpha=THRESHOLD_BELOW_ALPHA
)
idcs_blood = results_robust_blood.set_index(["cytokine", "celltype_combo"]).index
print(len(idcs_blood))
print(len(results_robust_blood.cytokine.unique()))

In [ ]:
all_results_csf = pd.read_csv("./results/all_results_enrichment_programs_csf.csv")
all_results_csf = all_results_csf.loc[all_results_csf.threshold_expression >= THRESHOLD_EXPRESSION]

all_results_csf = all_results_csf.loc[all_results_csf.celltype_combo != "NK1 (NK CD56hi)"]
all_results_csf = all_results_csf.loc[all_results_csf.celltype_combo != "NK2 (NK CD56low)"]
print(all_results_csf.loc[all_results_csf.percent_duplicate_ranking_stats < THRESHOLD_DUPLICATES].shape[0] / all_results_csf.shape[0])
all_results_csf = all_results_csf.loc[all_results_csf.percent_duplicate_ranking_stats < THRESHOLD_DUPLICATES]
results_pivot_csf, df_annot_csf, results_robust_csf, results_robust_csf_all = get_robust_significant_results(
    all_results_csf,
    alphas=[0.1, 0.05, 0.01],
    threshold_valid=THRESHOLD_VALID,
    threshold_below_alpha=THRESHOLD_BELOW_ALPHA
)
idcs_csf = results_robust_csf.set_index(["cytokine", "celltype_combo"]).index
print(len(idcs_csf))

In [ ]:
idcs = np.union1d(idcs_blood, idcs_csf)
print(len(idcs))

In [ ]:
missing_idcs = idcs[~np.isin(idcs, idcs_blood)].copy()
new_rows = pd.DataFrame(np.nan, columns=all_results_blood.set_index(["cytokine", "celltype_combo"]).columns, index=missing_idcs)
df_nes_blood = pd.concat(
        [
            all_results_blood.set_index(["cytokine", "celltype_combo"]), 
            new_rows
        ]
    ) \
    .reset_index() \
    .rename({"level_0": "cytokine", "level_1": "celltype_combo"}, axis=1) \
    .set_index(["cytokine", "celltype_combo"]) \
    .loc[idcs] \
    .groupby(["cytokine", "celltype_combo"])["NES"].mean() \
    .to_frame() \
    .reset_index()

df_nes_blood.loc[:, "CSF"] = False

In [ ]:
missing_idcs = idcs[~np.isin(idcs, idcs_csf)].copy()
new_rows = pd.DataFrame(np.nan, columns=all_results_csf.set_index(["cytokine", "celltype_combo"]).columns, index=missing_idcs)
df_nes_csf = pd.concat(
        [
            all_results_csf.set_index(["cytokine", "celltype_combo"]), 
            new_rows
        ]
    ) \
    .reset_index() \
    .rename({"level_0": "cytokine", "level_1": "celltype_combo"}, axis=1) \
    .set_index(["cytokine", "celltype_combo"]) \
    .loc[idcs] \
    .groupby(["cytokine", "celltype_combo"])["NES"].mean() \
    .to_frame() \
    .reset_index()

df_nes_csf.loc[:, "CSF"] = True

In [ ]:
# myeloid_compartment = np.array(["mDC1 (cDC)", "mDC2 (cDC)", "ncMono (CD16 Mono)", "Mono (CD14 Mono)"])
# myeloid_compartment = np.array(["mDC2 (cDC)", "ncMono (CD16 Mono)", "Mono (CD14 Mono)"])
myeloid_compartment = np.array(["mDC2 (cDC)", "ncMono (CD16 Mono)"])
lymphoid_compartment = np.array(['NK2 (NK CD56hi)', 'NK1 (NK CD56low)', 'CD4 (CD4 T cell)', 'CD8a (CD8 T cell)', 'CD8n (CD8 T cell)', 'B1 (B cell)', 'B2 (B cell)'])

programs_myeloid = np.union1d(
    results_robust_csf.loc[results_robust_csf.celltype_combo.isin(myeloid_compartment)].cytokine.unique(), 
    results_robust_blood.loc[results_robust_blood.celltype_combo.isin(myeloid_compartment)].cytokine.unique()
)

programs_lymphoid = np.union1d(
    results_robust_csf.loc[results_robust_csf.celltype_combo.isin(lymphoid_compartment)].cytokine.unique(), 
    results_robust_blood.loc[results_robust_blood.celltype_combo.isin(lymphoid_compartment)].cytokine.unique()
)

In [ ]:
figpath = "./results_final_DEGs/figures/"

### heatmaps for results on blood

In [ ]:
create_heatmap(all_results_blood, results_robust_blood, myeloid_compartment, results_robust_blood.cytokine.unique(), figsize=((2/3) * 0.75, 0.75), path=os.path.join(figpath, "enrichment_programs_myeloid_blood.svg"))
create_heatmap(all_results_blood, results_robust_blood, lymphoid_compartment, results_robust_blood.cytokine.unique(), figsize=(6/3 * 0.75, 6/4 * 0.75), path=os.path.join(figpath, "enrichment_programs_lymphoid_blood.svg"))

create_heatmap(all_results_blood, results_robust_blood, myeloid_compartment, programs_myeloid, figsize=((2/3) * 0.75, 5/4 * 0.75), path=os.path.join(figpath, "enrichment_programs_myeloid_blood2.svg"))
df_nes, df_annot = create_heatmap(all_results_blood, results_robust_blood, lymphoid_compartment, programs_lymphoid, figsize=(6/3 * 0.75, 7/4 * 0.75), path=os.path.join(figpath, "enrichment_programs_lymphoid_blood2.svg"))

In [ ]:
df_annot_T.columns

In [ ]:
df_nes_T = df_nes.T
df_annot_T = df_annot.T

order_y = [
    'CD8n (CD8 T cell)',
    'CD8a (CD8 T cell)',
    'NK1 (NK CD56low)',
    'NK2 (NK CD56hi)',
    'B1 (B cell)',
    'B2 (B cell)',
]

order_x = [
    'CE-03 Cytotoxic-2',
    'CE-01 Cytotoxic-1',
    'CE-06 EffectorTraffic',
    'CE-08 NK-Immuno',
    'FC-05 IgE-Humoral',
    'PD-03 BcellDiff-1',
    'PD-04 BcellReg',
]

df_nes_T = df_nes_T.loc[order_y, order_x]
df_annot_T = df_annot_T.loc[order_y, order_x]

fig, ax = plt.subplots(1, 1, figsize=(1.3, 1.3))
ax.set_facecolor("lightgray")
sns.heatmap(
    data=df_nes_T,
    annot=df_annot_T,
    fmt="",
    cmap="RdBu_r",
    square=True,
    annot_kws={"fontsize": 6, "family": "Arial"},
    ax=ax,
    cbar=False,
    cbar_kws={"shrink": 0.5},
    lw=0.5,
    linecolor="white",
    vmin=df_nes_T.min().min(),
    center=0,
    vmax=df_nes_T.max().max(),
)
ax.tick_params(axis='both', labelright=False, labelleft=True, length=0)
ax.set_yticks(0.5+np.arange(df_annot_T.shape[0]), format_celltype_combo_labels(df_annot_T.index), family="Arial", fontsize=6, rotation=0)
ax.set_xticks(0.5+np.arange(df_nes_T.shape[1]), format_program_labels(df_nes_T.columns), family="Arial", fontsize=6, rotation=90)
plt.xlabel("")
plt.ylabel("")
plt.savefig("./final_figures/heatmap_programs_blood_lymphoid.svg", bbox_inches="tight", pad_inches=0, transparent=False)
plt.show()

In [ ]:
def custom_cmap(cmap, vmin, vmax, center):
    cmap = plt.get_cmap(cmap)
    vrange = max(vmax - center, center - vmin)
    normlize = mcolors.Normalize(center - vrange, center + vrange)
    cmin, cmax = normlize([vmin, vmax])
    cc = np.linspace(cmin, cmax, 512)
    cmap2 = mcolors.ListedColormap(cmap(cc))
    return cmap2

vmin = df_nes_T.min().min()
vmax = df_nes_T.max().max()
center = 0
cmap = custom_cmap("RdBu_r", vmin=vmin, vmax=vmax, center=center)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
fig, ax = plt.subplots(figsize=(1.275, 0.08))
# fig.subplots_adjust(left=0.5, right=0.8)
cbar = colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal',)
cbar.ax.tick_params(labelsize=6)
cbar.ax.set_title("Norm. enrichment score", fontsize=6, family="Arial")
cbar.ax.spines[['right', 'left', 'top', 'bottom']].set_visible(False)
cbar.outline.set_visible(False)
cbar.ax.set_xticks([-1, 0, 1, 1.8], ["-1", "0", "1", "1.8"], family="Arial", fontsize=6)
cbar.ax.xaxis.set_ticks_position('top')
if True:
    plt.savefig("./final_figures/heatmap_programs_blood_lymphoid_colorbar.svg", bbox_inches="tight", pad_inches=0, transparent=True)
plt.show()

### heatmaps for results on CSF

In [ ]:
create_heatmap(all_results_csf, results_robust_csf, myeloid_compartment, results_robust_csf.cytokine.unique(), figsize=((2/3) * 0.75, (5/4) * 0.75), path=os.path.join(figpath, "enrichment_programs_myeloid_csf.svg"))
create_heatmap(all_results_csf, results_robust_csf, lymphoid_compartment, results_robust_csf.cytokine.unique(), figsize=((4/3) * 0.75, (4/4) * 0.75), path=os.path.join(figpath, "enrichment_programs_lymphoid_csf.svg"))

create_heatmap(all_results_csf, results_robust_csf, myeloid_compartment, programs_myeloid, figsize=((2/3) * 0.75, 5/4 * 0.75), path=os.path.join(figpath, "enrichment_programs_myeloid_csf2.svg"))
create_heatmap(all_results_csf, results_robust_csf, lymphoid_compartment, programs_lymphoid, figsize=(5/3 * 0.75, (6/7)*7/4 * 0.75), path=os.path.join(figpath, "enrichment_programs_lymphoid_csf2.svg"))